# Lab 09 · CUDA · write the GPU kernel yourself

Lab 08 offloaded with one OpenMP directive. This lab writes the same kernel in **CUDA C** — you allocate device memory, launch the kernel with `<<<grid, block>>>`, copy results back. You learn what the OpenMP `target` compiler was doing for you, and you get the low-level control to hand-tune what OpenMP won't.

**Prerequisites.** Lab 08 (OpenMP target running on Polaris). Familiar with kernels, blocks, warps at a conceptual level (skim Nvidia's programming guide chapter 2).

**Builds toward.** Lab 10 (multi-GPU with MPI+CUDA).

> **📚 Where to look when you're stuck**
>
> - [**CUDA C Programming Guide**](https://docs.nvidia.com/cuda/cuda-c-programming-guide/) — chapters 2 and 3 are essential
> - [**CUDA C Best Practices**](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/) — reference for tuning
> - [**Polaris CUDA setup**](https://docs.alcf.anl.gov/polaris/compiling-and-linking/) — module load nvhpc



## How this notebook works

Same three surfaces as prior labs: **[Hub]**, **[Hub -> cluster]**, **[cluster compute]**.


In [ ]:
# [Hub] Shared toolkit.
from labHelpers import *


### Set up this lab's identity


In [ ]:
# [Hub] Change HPC_USER; re-run.
env = setupLab(labName="lab09", host="polaris",
               remoteUser=os.environ.get("HPC_USER","CHANGE_ME"),
               project="UIC-CS455-Sp2027", queue="debug",
               scratch=f"/eagle/UIC-CS455-Sp2027/{os.environ.get('HPC_USER','CHANGE_ME')}")
labDir = pathlib.Path(env['labDir'])


### Preflight


In [ ]:
# [Hub] Reachability + prerequisite artifact.
preflight([
    check("passwordless ssh", sshReachable()),
    check("scheduler answers", schedulerAnswers()),
    check("lab09 dir on cluster", remoteFileExists(env['HPC_LAB_DIR']),
          hint="next cell creates it if missing"),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')),
             ('lab dir', env.get('HPC_LAB_DIR','?'))])


In [ ]:
# [Hub -> cluster] Make the lab dir if missing.
sshRun(f'mkdir -p {env["HPC_LAB_DIR"]}/out', quiet=True)
print('lab09 dir ready')


## Part 1 · The CUDA kernel

A **kernel** is a C function annotated `__global__`, called from the host, run in parallel on the GPU by many threads. Each thread computes its indices from `threadIdx`, `blockIdx`, `blockDim`, `gridDim`. Kernel launch syntax: `kernel<<<gridDim, blockDim>>>(args)`.

For our stencil, one thread per output cell: launch a 2D grid of 2D blocks covering the interior of the array.


In [ ]:
# [Hub] Write heat2D.cu.
(labDir/'heat2D.cu').write_text('''
/* heat2D.cu - CUDA version of the 2D heat stencil. */
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
static double wall(void){struct timespec t;clock_gettime(CLOCK_MONOTONIC,&t);return t.tv_sec+t.tv_nsec*1e-9;}
#define CHECK(x) do { cudaError_t e = (x); if (e!=cudaSuccess) { fprintf(stderr,"cuda err %s\\n",cudaGetErrorString(e)); exit(1);} } while(0)
__global__ void heatStep(const double *u, double *unew, int N, double a) {
    int i = blockIdx.y * blockDim.y + threadIdx.y + 1;
    int j = blockIdx.x * blockDim.x + threadIdx.x + 1;
    if (i >= N-1 || j >= N-1) return;
    int k = i*N + j;
    unew[k] = u[k] + a * (u[k-1] + u[k+1] + u[k-N] + u[k+N] - 4.0 * u[k]);
}
int main(int argc, char **argv){
    int N=2048, steps=500;
    for(int i=1;i<argc;i++){
        if(!strcmp(argv[i],"--N")&&i+1<argc) N=atoi(argv[++i]);
        if(!strcmp(argv[i],"--steps")&&i+1<argc) steps=atoi(argv[++i]);
    }
    size_t bytes = (size_t)N*N*sizeof(double);
    double *hU = (double*)calloc(N*N, sizeof(double));
    /* initial condition: hot square in center */
    for(int i=0;i<N;i++) for(int j=0;j<N;j++)
        if(i>0.4*N&&i<0.6*N&&j>0.4*N&&j<0.6*N) hU[i*N+j]=1.0;
    double *dU, *dUnew;
    CHECK(cudaMalloc(&dU, bytes)); CHECK(cudaMalloc(&dUnew, bytes));
    CHECK(cudaMemcpy(dU, hU, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(dUnew, hU, bytes, cudaMemcpyHostToDevice));
    dim3 block(32, 8);
    dim3 grid((N + block.x - 1)/block.x, (N + block.y - 1)/block.y);
    double a = 0.1 * 0.24 / 1.0;
    double t0 = wall();
    for(int s=0; s<steps; s++){
        heatStep<<<grid, block>>>(dU, dUnew, N, a);
        double *tmp = dU; dU = dUnew; dUnew = tmp;
    }
    CHECK(cudaDeviceSynchronize());
    double dt = wall() - t0;
    CHECK(cudaMemcpy(hU, dU, bytes, cudaMemcpyDeviceToHost));
    double sumU=0.0; for(long k=0;k<(long)N*N;k++) sumU += hU[k];
    double mlups = (double)N*N*steps/dt/1e6;
    printf("CUDA heat2D: N=%d steps=%d wall=%.3fs mlups=%.2f sumU=%.6e\\n",
           N, steps, dt, mlups, sumU);
    cudaFree(dU); cudaFree(dUnew); free(hU); return 0;
}
''')
showFile(labDir/'heat2D.cu', language='c', maxLines=25, title='heat2D.cu (top)')


In [ ]:
checkpoint("Part 1 - CUDA source", [
    check("heat2D.cu present", fileExists(str(labDir/'heat2D.cu'))),
])


## Part 2 · Build with `nvcc` and run

`nvcc` is Nvidia's CUDA compiler front-end. It splits your source into host code (compiled with the system C compiler) and device code (compiled to PTX/SASS for the GPU).


In [ ]:
# [Hub -> Polaris] Build + run.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
module load PrgEnv-nvhpc 2>/dev/null || module load nvhpc 2>/dev/null || true
nvcc -O3 -arch=sm_80 -o heat2Dcuda heat2D.cu
./heat2Dcuda --N 2048 --steps 500
'''
pbsPath = labDir/'cudaJob.pbs'
pbsPath.write_text(pbsHeader(name='lab09CUDA', project=env['HPC_PROJECT'],
                             queue='debug', select='1:system=polaris',
                             walltime='00:15:00', filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/cuda.out') + jobBody)
sshPut(str(labDir/'heat2D.cu'), env['HPC_LAB_DIR']+'/heat2D.cu')
sshPut(str(pbsPath),            env['HPC_LAB_DIR']+'/cudaJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/cudaJob.pbs'); waitJob(jobID, 30, 1200)
sshGet(env['HPC_LAB_DIR']+'/cuda.out', str(labDir/'cuda.out'))
print((labDir/'cuda.out').read_text())


In [ ]:
checkpoint("Part 2 - CUDA build+run", [
    check("cuda.out present", fileExists(str(labDir/'cuda.out'))),
    check("mlups reported", fileContains(str(labDir/'cuda.out'), 'mlups')),
])


## Part 3 · Compare to OMP `target`

Take lab 08's OMP-target MLUP/s and compare to your hand-written CUDA number. In most cases they'll be within ~20% of each other — the OMP compiler does a competent job on a regular kernel like this. When they diverge, usually CUDA wins because you got the block dimensions right by hand.


In [ ]:
checkpoint("Part 3 - CUDA vs OMP target", [
    check("cuda mlups printed", fileContains(str(labDir/'cuda.out'), 'mlups')),
])


## Part 4 · Block dimensions matter

The `dim3 block(32, 8)` in Part 1 was a guess. On an A100, warp = 32 threads, and the block.x should usually be 32 (or a multiple) so threads within a warp read contiguous memory. Try `block(32,4)`, `block(32,8)`, `block(32,16)`, `block(64,4)` and see which wins.

This is the CUDA equivalent of `OMP_SCHEDULE=` — a single knob that changes how work maps to hardware.


In [ ]:
# [Hub -> Polaris] Sweep block sizes.
print('Recompile with different `dim3 block(...)` values and rerun.')
print('For a 2D stencil the sweet spot on A100 is usually block(32, 8).')


In [ ]:
checkpoint("Part 4 - block-size understanding", [
    check("cuda.out available for reference", fileExists(str(labDir/'cuda.out'))),
])


## Part 5 · Correctness · does CUDA match the CPU?

Run the same N and steps as your lab 06 MPI run. The final `sumU` should match to ~1e-8 relative. Any bigger disagreement means a bug (usually an off-by-one in the boundary, or a stale halo if you added one).


In [ ]:
checkpoint("Part 5 - correctness", [
    check("cuda sumU printed", fileContains(str(labDir/'cuda.out'), 'sumU')),
])


## Part 6 · Bridge to lab 10

One GPU is fast. Many GPUs together is the state of the art. Lab 10 combines MPI (from lab 06) with CUDA (from this lab): each MPI rank owns one GPU, and halo exchanges go through **CUDA-aware MPI** so buffers on the GPU can be sent without a copy to host memory.


## Wrap up

Moved the spine forward one lab.


### Lab scorecard


In [ ]:
labSummary("CUDA")


---
### One-minute feedback

What worked, what didn't, what should be clearer.


In [ ]:
feedback("CUDA")
